## Setup


In [ ]:
!uv pip install "ase>=3.28.0" "upet>=0.2.1" "torch_dftd"

## Objective


Here, we will demonstrate how to model surfaces. We will study the adsorption of N2 on Fe. For this analysis, we will use the UPET-pet-mad-xs-r2SCAN machine learning potential with D3(BJ) dispersion corrections, as you will need to apply dispersion corrections in problem set 4. The choice of calculator can be easily swapped and is not the main focal point of this exercise.

The question we want to answer is: how strongly does N2 adsorb on the surface of iron?


We need to pick the dispersion (van der Waals) correction that matches the functional used in the training set. You can see on the [UPET website](https://github.com/lab-cosmo/upet) that the level of theory for the `pet-mad-xs` model is r2SCAN.


In [ ]:
from upet.calculator import UPETCalculator
from ase.calculators.mixing import SumCalculator
from torch_dftd.torch_dftd3_calculator import TorchDFTD3Calculator

device = "cpu"  # or "cuda"
upet = UPETCalculator(model="pet-mad-xs", version="1.5.0", device=device)
dft_d3 = TorchDFTD3Calculator(device=device, xc="r2scan", damping="bj")
calc = SumCalculator([upet, dft_d3])

## Optimizing and Carving the Surface Structure


Our first order of business is to construct a surface of Fe. The bulk structure of Fe could be obtained from an experimental crystal structure, from the Materials Project, or any other place. We will take the one provided by ASE for convenience. You could then carve this bulk surface using various tools in ASE. For simplicitly, I will use an ASE tool that can directly get the fcc 111 surface of Fe.


In [ ]:
from ase.build import fcc111
from ase.constraints import FixAtoms

slab = fcc111("Fe", size=(6, 6, 4), a=3.323, vacuum=10.0)

c = FixAtoms(mask=[atom.tag > 2 for atom in slab])
slab.set_constraint(c)

In [ ]:
from ase.visualize import view

view(slab, viewer="x3d")

Now we should relax this surface to find its lowest energy configuration. In pratice, it is more wise to try many carvings of the bulk Fe and calculate the surface energy of many surfaces. The surface with the lowest surface energy is then the most stable surface. For time and simplicitly, we will assum fcc111 is the most stable configuration.


In [ ]:
from ase.filters import FrechetCellFilter
from ase.optimize import BFGS

slab.calc = calc  # Assign a calculator
slab_wrapped = FrechetCellFilter(slab)  # Tell ASE to optimize cell too
opt = BFGS(slab_wrapped, trajectory="slab_relax.traj")  # Set up optimizer
opt.run(fmax=0.01)  # Run optimization until forces < 0.01 eV/Å

## Adding Adsorbates


Let's start by defining and relaxing the isolated N2 molecule.


In [ ]:
from ase.build import molecule

adsorbate = molecule("N2")
adsorbate.calc = calc
opt = BFGS(adsorbate)
opt.run(fmax=0.01)

Now we need to add our adsorbate to our surface. There are many possible surface sites. The only way to know where the adsorbate should go is to add the adsorbate to all plausible surface sites, do a structure relaxation, and identify the lowest energy configuration(s). There are tools in Pymatgen to do this. I will show you a simplified example of how to do this using the ase gui, as this will be the easiest method for your PS4 homework assingment.


- use Edit -> Add atoms and type in "N2" under "Add:", deselect check positions
- move N2 molecule around with Tools -> Move selected atoms
- save this structure with File -> Save
- NOTE: To keep the atom constraints in the bulk atoms of this new file, it is easiest to save this as a VASP file like a POSCAR. If you save this crystal structure as a .cif, you will need to set the constraints on the bulk atoms again, since .cif files do not contain information on atom constraints


In [ ]:
view(slab)

Now that we have the initial configuration of our slab + adsorbate, we need to relax it once again.


In [ ]:
from ase.io import read

slab_with_N2 = read("POSCAR")
slab_with_N2.calc = calc
opt = BFGS(slab_with_N2)
opt.run(fmax=0.01)

In [ ]:
view(slab_with_N2, viewer="x3d")

In practice, it is necessary to enumerate all sites for N2 adsorption and find the configuration with the lowest total energy. For simplicity we will assume we found the most stable site.


## Calculating the Adsorption Energy


In [ ]:
delta_E = (
    slab_with_N2.get_potential_energy()
    - slab.get_potential_energy()
    - adsorbate.get_potential_energy()
)
print(f"Adsorption energy: {delta_E} eV")

Experimentally, it is known that Fe will dissociate N2 under typical reaction conditions. At low temperatures (~140 K), molecular adsorption (i.e. without dissociation) occurs. For reference, on the Fe(111) surface, the N2 adsorption energy is estimated to be between -0.22 eV to -0.43 eV. Source: https://doi.org/10.1016/0021-9517(77)90237-8.


## Exercise


Try adsorbing a different molecule other than N2 on the surface of Fe fcc111. What is its adsorption energy?
